# Desafío Data Scientist MRNF: Modelado Predictivo y Optimización Financiera
**Área:** Modelos de Riesgo No Financiero (MRNF)  
**Objetivo:** Desarrollar, comparar y optimizar modelos de clasificación para detección de fraude transaccional maximizando la ganancia económica neta del emisor.  
**Función de Utilidad Financiera:**  
$$\text{Ganancia Neta} = (120 \times \text{VP}) - (30 \times \text{FP}) - (180 \times \text{FN}) + (0 \times \text{VN})$$
- **VP (+US$ 120):** Fraude prevenido, neto del costo de alerta.
- **FP (-US$ 30):** Fricción operativa, llamada a call center y molestia de cliente.
- **FN (-US$ 180):** Fraude consumado, contracargo y reposición del plástico.
- **VN (US$ 0):** Transacción legítima aprobada sin costo.

## 1. Carga de Datos Procesados y División Temporal Out-of-Time

In [ ]:
import sys
from pathlib import Path

import pandas as pd
import polars as pl

current_dir = Path.cwd().resolve()
PROJECT_ROOT = current_dir.parent if current_dir.name == "notebooks" else current_dir
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import (
    PROCESSED_DATA_PATH,
    TARGET_COL,
    TEST_OOT_END_DT,
    TEST_OOT_START_DT,
    TRAIN_START_DT,
    VAL_START_DT,
)
from src.models.evaluation import (
    compute_detailed_metrics,
    compute_financial_utility,
    find_optimal_business_threshold,
    plot_business_utility_tradeoff,
    plot_confusion_matrix,
    plot_precision_recall_curves,
    plot_roc_curves,
)
from src.models.preprocessing import ALL_MODEL_FEATURES, create_preprocessor
from src.utils.plots import plot_feature_importance

print("Cargando transacciones con variables enriquecidas...")
df = pl.read_parquet(PROCESSED_DATA_PATH)

# Split estrictamente Out-of-Time
train_df = df.filter(
    (pl.col("timestamp") >= TRAIN_START_DT) & (pl.col("timestamp") < VAL_START_DT)
)
val_df = df.filter(
    (pl.col("timestamp") >= VAL_START_DT) & (pl.col("timestamp") < TEST_OOT_START_DT)
)
test_df = df.filter(
    (pl.col("timestamp") >= TEST_OOT_START_DT)
    & (pl.col("timestamp") <= TEST_OOT_END_DT)
)

print(
    f"Train set: {train_df.height:,} tx (Fraude: {train_df[TARGET_COL].sum():,}, {train_df[TARGET_COL].mean() * 100:.3f}%)"
)
print(
    f"Val set:   {val_df.height:,} tx (Fraude: {val_df[TARGET_COL].sum():,}, {val_df[TARGET_COL].mean() * 100:.3f}%)"
)
print(
    f"Test OOT:  {test_df.height:,} tx (Fraude: {test_df[TARGET_COL].sum():,}, {test_df[TARGET_COL].mean() * 100:.3f}%)"
)

## 2. Construcción del Pipeline de Preprocesamiento y Ajuste

In [ ]:
X_train = train_df.select(ALL_MODEL_FEATURES).to_pandas()
y_train = train_df[TARGET_COL].to_pandas()

X_val = val_df.select(ALL_MODEL_FEATURES).to_pandas()
y_val = val_df[TARGET_COL].to_pandas()

X_test = test_df.select(ALL_MODEL_FEATURES).to_pandas()
y_test = test_df[TARGET_COL].to_pandas()

preprocessor = create_preprocessor()
Xt_train = preprocessor.fit_transform(X_train, y_train)
Xt_val = preprocessor.transform(X_val)
Xt_test = preprocessor.transform(X_test)

feature_names = preprocessor.get_feature_names_out().tolist()
print(f"Dimensiones de features transformadas: {Xt_train.shape[1]} variables")

## 3. Benchmarking de Modelos Predictivos en Validación

In [ ]:
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

models = {
    "LightGBM Baseline": LGBMClassifier(
        n_estimators=50,
        max_depth=3,
        learning_rate=0.1,
        random_state=42,
        n_jobs=4,
        verbose=-1,
    ),
    "LightGBM (Tuned)": LGBMClassifier(
        n_estimators=200,
        max_depth=6,
        num_leaves=31,
        learning_rate=0.05,
        scale_pos_weight=5.0,
        random_state=42,
        n_jobs=4,
        verbose=-1,
    ),
    "XGBoost (Champion)": XGBClassifier(
        n_estimators=200,
        max_depth=5,
        learning_rate=0.05,
        scale_pos_weight=5.0,
        random_state=42,
        n_jobs=4,
        eval_metric="logloss",
    ),
}

val_results = {}
from sklearn.metrics import average_precision_score, roc_auc_score

from src.models.trainer import ModelEvaluationResult

for name, model in models.items():
    print(f"Ajustando {name}...")
    model.fit(Xt_train, y_train)
    proba = model.predict_proba(Xt_val)[:, 1]
    pr = average_precision_score(y_val, proba)
    roc = roc_auc_score(y_val, proba)
    val_results[name] = ModelEvaluationResult(name, model, pr, roc, 0, 0, 0, 0, proba)
    print(f"  -> PR-AUC: {pr:.4f} | ROC-AUC: {roc:.4f}")

## 4. Curvas de Rendimiento Diagnóstico (PR y ROC)

In [ ]:
fig_pr = plot_precision_recall_curves(val_results, y_val.to_numpy())
fig_pr.show()

fig_roc = plot_roc_curves(val_results, y_val.to_numpy())
fig_roc.show()

## 5. Optimización del Umbral de Corte por Utilidad Financiera Neta

In [ ]:
champion_proba = val_results["XGBoost (Champion)"].y_proba
th_tuning = find_optimal_business_threshold(y_val.to_numpy(), champion_proba)

print("=== RESULTADOS DE OPTIMIZACIÓN FINANCIERA (VALIDACIÓN) ===")
print(f"Umbral Óptimo Financiero: {th_tuning.optimal_threshold:.3f}")
print(f"Utilidad Neta Máxima: US$ {th_tuning.max_utility:,.0f}")
print(f"Verdaderos Positivos: {th_tuning.tp_at_optimal} (de {y_val.sum()} fraudes)")
print(f"Falsos Positivos:     {th_tuning.fp_at_optimal}")
print(f"Falsos Negativos:     {th_tuning.fn_at_optimal}")

fig_util = plot_business_utility_tradeoff(th_tuning, model_name="XGBoost Champion")
fig_util.show()

## 6. Evaluación Definitiva Out-of-Time (Mayo 2026)

In [ ]:
champion_model = models["XGBoost (Champion)"]
p_test = champion_model.predict_proba(Xt_test)[:, 1]
optimal_th = th_tuning.optimal_threshold

metrics_default = compute_detailed_metrics(y_test.to_numpy(), p_test, threshold=0.5)
metrics_optimal = compute_detailed_metrics(
    y_test.to_numpy(), p_test, threshold=optimal_th
)
no_model_util = compute_financial_utility(
    0, 0, int(y_test.sum()), len(y_test) - int(y_test.sum())
)

comparison_df = pd.DataFrame(
    [
        {
            "Escenario": "Sin Modelo (Inacción)",
            "Umbral": "-",
            "Fraudes Detectados (VP)": 0,
            "Falsas Alarmas (FP)": 0,
            "Fraudes Omitidos (FN)": int(y_test.sum()),
            "Utilidad Neta (US$)": no_model_util,
            "Ahorro vs Inacción (US$)": 0,
        },
        {
            "Escenario": "Modelo Default",
            "Umbral": "0.500",
            "Fraudes Detectados (VP)": metrics_default["TP"],
            "Falsas Alarmas (FP)": metrics_default["FP"],
            "Fraudes Omitidos (FN)": metrics_default["FN"],
            "Utilidad Neta (US$)": metrics_default["Financial_Utility_USD"],
            "Ahorro vs Inacción (US$)": metrics_default["Financial_Utility_USD"]
            - no_model_util,
        },
        {
            "Escenario": "Modelo Optimizado",
            "Umbral": f"{optimal_th:.3f}",
            "Fraudes Detectados (VP)": metrics_optimal["TP"],
            "Falsas Alarmas (FP)": metrics_optimal["FP"],
            "Fraudes Omitidos (FN)": metrics_optimal["FN"],
            "Utilidad Neta (US$)": metrics_optimal["Financial_Utility_USD"],
            "Ahorro vs Inacción (US$)": metrics_optimal["Financial_Utility_USD"]
            - no_model_util,
        },
    ]
)

display(comparison_df)

fig_cm = plot_confusion_matrix(
    metrics_optimal["confusion_matrix"],
    optimal_th,
    model_name="XGBoost Champion (Test OOT)",
)
fig_cm.show()

## 7. Importancia de Variables e Interpretabilidad

In [ ]:
fig_fi = plot_feature_importance(
    feature_names,
    champion_model.feature_importances_,
    top_n=15,
    title="Top 15 Características más Predictivas (XGBoost)",
)
fig_fi.show()

Hallazgos Clave:
1. ratio_amount_mean_7d y card_tx_count_1h lideran el poder predictivo, capturando desviaciones abruptas de consumo.
2. is_country_mismatch y cards_seen_by_device_24h blindan contra ataques automatizados transfronterizos.
3. is_night_transaction y mcc_encoded canalizan el riesgo horario y sectorial.